In [81]:
#import serpapi
import pandas as pd
import glob
import random
import webbrowser

In [ ]:
langs = pd.read_csv('Lang_List.csv', delimiter=';')
remaining_music = list(langs["Name"])[7038:] 
remaining_families = list(langs["Families"])[7038:]

# API Approach

In [2]:
def lang_music_csv(music):
    for i in range(len(music)):
        client = serpapi.Client(api_key="311a68e2f949414ba3da2f87bc9795c37df6638e0a8c056483bc0720fe79c8b3")
        results = client.search({
            "engine": "google",
            "q": music[i]
        })
        organic_results = results["organic_results"]

        titles = []
        links = []
        snippets = []

        for result in organic_results:
            title = result["title"]
            link = result["link"]
            snippet = result["snippet"]

            titles.append(title)
            links.append(link)
            snippets.append(snippet)

        titles_df = pd.DataFrame(
            data=titles, 
            columns=["Title"]
        )

        links_df = pd.DataFrame(
            data=links, 
            columns=["Link"]
        )

        snippets_df = pd.DataFrame(
            data=snippets, 
            columns=["Snippet"]
        )

        results_df = pd.concat([titles_df, links_df, snippets_df], axis=1)
        results_df.to_csv(f"{music[i]}_music_google.csv")

        i += 1

In [7]:
lang_music_csv(remaining_music)

HTTPError: 429 Client Error: Too Many Requests for url: https://serpapi.com/search?engine=google&q=Andio&api_key=311a68e2f949414ba3da2f87bc9795c37df6638e0a8c056483bc0720fe79c8b3

# Pipeline Approach

## 1. Retrieving URLs

## Figure out ddgs installation

In [2]:
import string
from ddgs import DDGS

with DDGS(timeout=120) as ddgs:
    for language, family in zip(remaining_music, remaining_families):

        query = f"{language} {family} music"

        results = list(ddgs.text(query, max_results=10))

        df = pd.DataFrame(results)

        df["language"] = language
        df["family"] = family

        df.rename(columns={
            "href": "link",
            "body": "text"
        }, inplace=True)

        df.to_csv(f"{language.translate(str.maketrans('', '', string.punctuation))}.csv", index=False)

## Finding language at Timeout

In [29]:
langs.iloc[langs["Name"] == "Wawa"]

,Name,Families,Unnamed: 2,Unnamed: 3
7037,Wawa,Niger-Congo,NaN,NaN


In [ ]:
langs.iloc[4605]

Name          Naga, Khiamniungan
Families            Sino-Tibetan
Unnamed: 2                   NaN
Unnamed: 3                   NaN
Name: 4604, dtype: object

## Dask and Coiled to scale CSV reads

import coiled
cluster = coiled.Cluster(
    n_workers=100,
)
client = cluster.get_client()

import dask.dataframe as dd

df = dd.read_parquet("Music_results")

Need to iterate through file path
Then use Coiled to scale the project
Dask is for processing multiple dataframes at once

In [47]:
mega_lang_csv_list = glob.glob('Music_results/*.csv')

In [53]:
pd.read_csv(mega_lang_csv_list[2])

,title,link,text,language,family
0,Afroasiatic Linguistic Features and Typologies...,https://www.cambridge.org/core/books/cambridge...,"Afroasiatic (occasionally also Afro-Asiatic), ...",Aasáx,Afro-Asiatic
1,Chadic languages a subdivision of Afro-Asiatic...,https://www.facebook.com/afrikarmu/posts/chadi...,"Sep 23, 2024 ... Beside the possibilities list...",Aasáx,Afro-Asiatic
2,George E. Lewis - COLUMBIA | MUSIC,https://music.columbia.edu/bios/george-e-lewis,"Lewis, eds., Composing While Black: Afrodiaspo...",Aasáx,Afro-Asiatic
3,The Afro-Asiatic Languages: Classification and...,https://www.yumpu.com/en/document/view/2582743...,"Jan 9, 2014 ... South Rift East Asax †. Kw'adz...",Aasáx,Afro-Asiatic
4,African languages share common roots and diale...,https://www.facebook.com/groups/86494001758437...,"Dec 18, 2021 ... Hausa(Afro-Asiatic) Swahili(N...",Aasáx,Afro-Asiatic
5,Gorwaa (Tanzania) - Language Documentation and...,https://www.lddjournal.org/article/1200/galley...,Gorwaa is a member of the Southern Cushitic gr...,Aasáx,Afro-Asiatic
6,ComparaLex,https://comparalex.org/,Afro-Asiatic. Berber. Eastern. Awjila-Sokna. A...,Aasáx,Afro-Asiatic
7,On the Town (1944) - Works | Works | Leonard B...,https://leonardbernstein.com/works/view/8/on-t...,"Everett Lee, the show's conductor made history...",Aasáx,Afro-Asiatic
8,Understanding Abadi Language: An Endangered Tr...,https://www.tiktok.com/@linguadex/video/756407...,"Oct 22, 2025 ... The Asa language, commonly re...",Aasáx,Afro-Asiatic
9,iso-languagecodes.txt - GeoNames,http://download.geonames.org/export/dump/iso-l...,... Afro-Asiatic languages alg Algonquian lang...,Aasáx,Afro-Asiatic


In [57]:
links_dfs = []
for file in mega_lang_csv_list:
    link_df = pd.read_csv(file, on_bad_lines='skip')
    links_dfs.append(link_df)

In [64]:
mega_lang_df = pd.concat(links_dfs, ignore_index=True)

In [69]:
mega_lang_df = mega_lang_df.drop(mega_lang_df.columns[5:], axis=1)

In [70]:
mega_lang_df.to_csv("mega_lang_list.csv")

## Random Number Generator for Opening Link

In [82]:
random_num = random.randint(0, len(mega_lang_df))
random_link = mega_lang_df.iloc[random_num]["link"]
webbrowser.open(random_link)

True

## Testing classify just music

In [2]:
import pandas as pd

mega_lang_df = pd.read_csv("D:/Languages_Project/Languages_Project/mega_lang_list.csv")

In [4]:
snippet = mega_lang_df.head(5)
snippet.to_csv("snippet.csv")